# Lab 04: Supervised Learning — Linear Regression, Logistic Regression, and Decision Trees
**Week 5 — October 12**

A restaurant manager wants to understand tipping. In this lab you will use a classic practice dataset, in which one waiter recorded details of each table he served, to build three supervised models: one that predicts the **dollar amount** of a tip, one that predicts whether a tip is **generous**, and a decision tree that does the second job with a sequence of yes/no questions. The data come from one waiter in one restaurant, so do not generalize the results to all restaurants.

By the end, you should be able to:
- Decide whether a problem calls for regression or classification, and prepare features (one-hot encoding and a train/test split).
- Fit a linear regression, evaluate it against a baseline using MSE and R², and interpret a coefficient.
- Fit a logistic regression, read predicted probabilities, and evaluate it with accuracy, precision, and recall.
- Fit a decision tree and explain how `max_depth` controls overfitting.
- Compare the three models and say when each is appropriate.

Work from top to bottom. Run cells with **Shift + Enter**. Replace every **...** in an answer cell and complete the written responses. You can work with a neighbor and ask a tutor/TA for help.

## Setup

The dataset is loaded from **tips.csv**, included with this lab. Each row is one table served by the waiter.

| Column | Meaning |
|---|---|
| total_bill | Total bill in dollars |
| tip | Tip in dollars |
| sex | Sex of the person paying the bill |
| smoker | Whether the party included smokers |
| day | Day of the week (Thur, Fri, Sat, Sun) |
| time | Lunch or Dinner |
| size | Number of people at the table |

The setup cell also creates one new column, **generous**: it is 1 when the tip is at least 15% of the bill and 0 otherwise. We will predict **tip** (a number) for regression and **generous** (a category) for classification.

> **Watch out for leakage.** `tip` is a target, and `generous` is computed from `tip`. Neither may be used as an input feature, or the model would be handed the answer.

### The three models at a glance

| Model | What it learns | Predicts | Output | scikit-learn class |
|---|---|---|---|---|
| **Linear regression** | A weighted sum of the features | A number | A number (tip in dollars) | `LinearRegression` |
| **Logistic regression** | A weighted sum, squashed into a probability | A category (0 or 1) | A probability, then a class | `LogisticRegression` |
| **Decision tree** | A sequence of yes/no questions | A category (or a number) | A class | `DecisionTreeClassifier` |

In [ ]:
import numpy as np
import pandas as pd
import matplotlib.pyplot as plt
from sklearn.model_selection import train_test_split
from sklearn.linear_model import LinearRegression, LogisticRegression
from sklearn.tree import DecisionTreeClassifier, plot_tree
from sklearn.metrics import (mean_squared_error, r2_score, accuracy_score,
                             precision_score, recall_score, confusion_matrix)
import otter

grader = otter.Notebook("Lab04.ipynb", tests_dir="tests/lab04")

tips = pd.read_csv("tips.csv")
category_orders = {
    "sex": ["Male", "Female"],
    "smoker": ["Yes", "No"],
    "day": ["Thur", "Fri", "Sat", "Sun"],
    "time": ["Lunch", "Dinner"],
}
for column, categories in category_orders.items():
    tips[column] = pd.Categorical(tips[column], categories=categories)
tips["generous"] = (tips["tip"] / tips["total_bill"] >= 0.15).astype(int)

print("Rows:", len(tips))
print("Share of generous tips:")
print(tips["generous"].value_counts(normalize=True).round(3))
tips.head()

### Q1. What kind of learning is this?

Predicting the dollar **tip** from the other columns is supervised **regression**. Explain why it is supervised and why it is regression. Then classify these two tasks:

1. Predict whether a table leaves a **generous** tip, using past tables labeled generous / not generous.
2. Group tables by party size and bill amount **without** any label to predict.

> Write your answer here.

## Part 1: Prepare the data

Models need numerical inputs, and we need held-out rows to judge them honestly.

### Q2. One-hot encode the features

Save a DataFrame named **X** by one-hot encoding the categorical columns of `tips[feature_columns]`. Numeric columns pass through unchanged. Use `drop_first=True` so each category variable gets one fewer column than it has categories (the dropped category is implied when all the others are 0, which avoids redundant columns), and `dtype=int`.

Do **not** include `tip` or `generous`. You should end up with 8 columns.

**Hint:** `pd.get_dummies(data, drop_first=True, dtype=int)`.

In [ ]:
feature_columns = ["total_bill", "size", "sex", "smoker", "day", "time"]
X = ...
X.head()

In [ ]:
grader.check("q2")

### Q3. Split into training and test sets

We have **two** targets: `tips["tip"]` (regression) and `tips["generous"]` (classification). Split everything in one call so the rows stay aligned. `train_test_split` accepts any number of arrays and returns a train piece and a test piece for each, in order. Hold out 20% of rows for testing and use `random_state=42`.

**Hint:** `train_test_split(X, tips["tip"], tips["generous"], test_size=..., random_state=...)` returns `X_train, X_test, y_train, y_test, g_train, g_test`. Here `y` is the tip amount and `g` is the generous label.

In [ ]:
X_train, X_test, y_train, y_test, g_train, g_test = train_test_split(
    ..., ..., ..., test_size=..., random_state=...)

print("Training rows:", len(X_train), "| Test rows:", len(X_test))

In [ ]:
grader.check("q3")

## Part 2: Linear regression

**Linear regression** predicts a number as a weighted sum of the features:

$$\widehat{\text{tip}} = b + w_1\,(\text{total\_bill}) + w_2\,(\text{size}) + \dots$$

The intercept $b$ and the weights $w_j$ (the **coefficients**) are the parameters. Training finds the values that minimize the MSE on the training rows. Each coefficient says how much the prediction changes when that feature goes up by one unit **while the other features stay fixed**.

### Q4. Fit the model and predict

Create a **LinearRegression** model named **lin_model**, fit it on the **training** rows (`X_train`, `y_train`), and save its predictions for the **test** rows as **lin_pred**.

**Hint:** Create with `LinearRegression()`, train with `.fit(X_train, y_train)`, predict with `.predict(X_test)`. Never fit on test rows.

In [ ]:
lin_model = ...
lin_model.fit(..., ...)
lin_pred = ...
lin_pred[:5]

In [ ]:
grader.check("q4")

### Q5. Evaluate against a baseline

A model is only useful if it beats a simple guess. Our baseline predicts the **mean training tip** for every test row. Compute three numbers on the **test** rows:

- **baseline_mse**: the MSE of the baseline.
- **lin_mse**: the MSE of the linear model.
- **lin_r2**: the R² of the linear model. An R² of 1 is perfect, 0 means no better than always predicting the mean tip, and negative is worse than that.

The error for each row is **actual − predicted**, and $\mathrm{MSE}=\frac{1}{n}\sum_i (y_i-\hat y_i)^2$. MSE is in squared dollars, so it is not directly a dollar amount.

**Hint:** For the baseline use `np.mean((y_test - y_train.mean()) ** 2)`. Use the **training** tips for the mean, because we would not know the test tips in advance. For the others, use `mean_squared_error(actual, predicted)` and `r2_score(actual, predicted)`.

In [ ]:
baseline_mse = ...
lin_mse = ...
lin_r2 = ...
print("Baseline MSE:", round(baseline_mse, 3))
print("Linear regression MSE:", round(lin_mse, 3))
print("Linear regression R²:", round(lin_r2, 3))

In [ ]:
grader.check("q5")

Run the plot below after Q5. Points on the dashed line would be perfect predictions.

In [ ]:
plt.scatter(y_test, lin_pred)
limit = max(y_test.max(), lin_pred.max()) + 0.5
plt.plot([0, limit], [0, limit], "--", color="gray", label="Perfect predictions")
plt.xlabel("Actual tip ($)")
plt.ylabel("Predicted tip ($)")
plt.title("Linear regression: predicted vs. actual tips (test rows)")
plt.legend()
plt.show()

**Your response:** Did the linear model beat the baseline? Use both MSE values and the R² in your answer. Does the plot suggest the predictions are accurate for the largest tips?

> Write your answer here.

### Q6. Look at the coefficients

Save the learned coefficients as a **pandas Series** named **lin_coefs**, with the column names of `X` as its index so each number is labeled.

**Hint:** `pd.Series(lin_model.coef_, index=X.columns)`.

In [ ]:
lin_coefs = ...
lin_coefs.round(3)

In [ ]:
grader.check("q6")

**Your response:** (a) Interpret the **total_bill** coefficient in one sentence, with units. (b) The **size** coefficient is larger than the **total_bill** coefficient. Why can't we conclude that party size matters more than the bill? (c) Why does this model not prove that bigger bills *cause* bigger tips?

> Write your answer here.

## Part 3: Logistic regression

Despite its name, **logistic regression** is a **classification** model. It computes the same kind of weighted sum $z = b + w_1x_1 + w_2x_2 + \dots$, then squashes it into a probability between 0 and 1 with the **sigmoid** function:

$$p = P(\text{generous}) = \frac{1}{1 + e^{-z}}$$

By default the model predicts class 1 when $p \ge 0.5$ and class 0 otherwise. Run the cell below to see the S-shaped curve.

In [ ]:
z = np.linspace(-6, 6, 200)
plt.plot(z, 1 / (1 + np.exp(-z)), lw=2.5)
plt.axhline(0.5, color="gray", linestyle="--", label="Default threshold (0.5)")
plt.xlabel("Weighted sum  z")
plt.ylabel("Probability  p")
plt.title("The sigmoid turns any score into a probability")
plt.legend()
plt.show()

### Q7. Fit the model and get probabilities

Create a **LogisticRegression** model named **log_model** with `max_iter=1000`, fit it on the **training** rows (`X_train`, `g_train`), and save two sets of results for the **test** rows:

- **log_pred**: the predicted class (0 or 1).
- **log_proba**: the predicted probability of class 1 (generous).

**Hint:** `.predict_proba(X_test)` returns **two** columns: the probability of class 0, then of class 1. Keep only the second column with `[:, 1]`.

In [ ]:
log_model = ...
log_model.fit(..., ...)
log_pred = ...
log_proba = ...
print("First 5 predicted classes:", log_pred[:5])
print("First 5 probabilities of 'generous':", log_proba[:5].round(2))

In [ ]:
grader.check("q7")

### Q8. Evaluate the classifier

Accuracy alone can mislead, so compute four numbers on the **test** rows:

- **majority_acc**: the accuracy of a baseline that always predicts the most common class in the **training** labels.
- **log_acc**: the accuracy of the logistic regression: the fraction of predictions that are correct.
- **log_precision**: of the tables predicted generous, the fraction that really were generous. $\frac{TP}{TP+FP}$
- **log_recall**: of the tables that really were generous, the fraction the model caught. $\frac{TP}{TP+FN}$

**Hint:** `majority_class = g_train.mode()[0]` gives the most common training label. The baseline accuracy is the fraction of `g_test` equal to it. Use `accuracy_score`, `precision_score`, and `recall_score`, each called as `(actual, predicted)`.

In [ ]:
majority_class = g_train.mode()[0]
majority_acc = ...
log_acc = ...
log_precision = ...
log_recall = ...
print("Majority-class baseline accuracy:", round(majority_acc, 3))
print("Logistic regression accuracy:", round(log_acc, 3))
print("Precision:", round(log_precision, 3), "| Recall:", round(log_recall, 3))

In [ ]:
grader.check("q8")

Run the cell below after Q8 to see the confusion matrix: how many test tables fell into each combination of actual and predicted class.

In [ ]:
pd.DataFrame(confusion_matrix(g_test, log_pred),
             index=["Actually not generous", "Actually generous"],
             columns=["Predicted not generous", "Predicted generous"])

**Your response:** (a) Did the logistic regression beat the majority-class baseline? (b) In the words of this problem, what does a **false positive** mean, and what does a **false negative** mean? (c) The test set has only 49 rows. How should that change how much you trust the accuracy?

> Write your answer here.

### Q9. Move the decision threshold

The 0.5 cutoff is a choice, not a law. The next cell re-classifies the same test rows using three different thresholds on `log_proba`. Run it and compare.

In [ ]:
for threshold in [0.3, 0.5, 0.7]:
    pred = (log_proba >= threshold).astype(int)
    print(f"Threshold {threshold}: predicted generous = {pred.sum():2d} | "
          f"precision = {precision_score(g_test, pred, zero_division=0):.2f} | "
          f"recall = {recall_score(g_test, pred):.2f}")

**Your response:** What happens to the number of tables predicted generous, to precision, and to recall as the threshold rises from 0.3 to 0.7? Describe a situation where you would prefer a **high** threshold and one where you would prefer a **low** threshold.

> Write your answer here.

## Part 4: Decision trees

A **decision tree** predicts by asking a sequence of yes/no questions about the features, such as "is total_bill ≤ 26?" Each question splits the rows into groups; the tree chooses splits that make the groups as "pure" (dominated by one class) as possible. A **leaf** at the bottom gives the final prediction.

The most important setting is **max_depth**, the maximum number of questions along any path. A shallow tree is simple and may **underfit**; a very deep tree can create a leaf for almost every training row and **overfit** (memorize the training data).

### Q10. Fit a shallow tree

Create a **DecisionTreeClassifier** named **tree_model** with `max_depth=3` and `random_state=42`, fit it on the **training** rows (`X_train`, `g_train`), and save its accuracy on the **test** rows as **tree_acc**.

**Hint:** For a classifier, `model.score(X_test, g_test)` returns the accuracy.

In [ ]:
tree_model = ...
tree_model.fit(..., ...)
tree_acc = ...
print("Tree test accuracy:", round(tree_acc, 3))
print("Majority-class baseline:", round(majority_acc, 3))

In [ ]:
grader.check("q10")

Run the cell below to draw the tree you just trained and to see how much each feature was used for splitting (**feature importance**; the values sum to 1).

In [ ]:
plt.figure(figsize=(14, 6))
plot_tree(tree_model, feature_names=list(X.columns), class_names=["Not generous", "Generous"],
          filled=True, rounded=True, fontsize=9)
plt.title("Decision tree (max_depth = 3)")
plt.show()

pd.Series(tree_model.feature_importances_, index=X.columns).sort_values(ascending=False).round(3)

**Your response:** Read the **root node** (the first question at the top). What does it ask? Which feature did the tree rely on most, and how does that compare with what you saw in the models above?

> Write your answer here.

### Q11. How deep is too deep?

Train one tree for each depth in the list below, always with `random_state=42` on the **training** rows. For each, record the accuracy on the training rows (**train_acc**) and on the test rows (**test_acc**). The loop collects the results into a DataFrame named **depth_results**; fill in the two blanks.

**Hint:** Use `tree.score(X_train, g_train)` and `tree.score(X_test, g_test)`.

In [ ]:
results = []
for depth in [1, 2, 3, 5, 8, 12]:
    tree = DecisionTreeClassifier(max_depth=depth, random_state=42)
    tree.fit(X_train, g_train)
    train_acc = ...
    test_acc = ...
    results.append({"max_depth": depth, "train_acc": train_acc, "test_acc": test_acc})

depth_results = pd.DataFrame(results)
depth_results.round(3)

In [ ]:
grader.check("q11")

Run the cell below after Q11. It adds a **gap** column (training accuracy minus test accuracy) and plots both accuracies against depth.

In [ ]:
depth_results["gap"] = depth_results["train_acc"] - depth_results["test_acc"]
display(depth_results.round(3))

plt.plot(depth_results["max_depth"], depth_results["train_acc"], "o-", label="Training accuracy")
plt.plot(depth_results["max_depth"], depth_results["test_acc"], "s-", label="Test accuracy")
plt.axhline(majority_acc, color="gray", linestyle=":", label="Majority-class baseline")
plt.xlabel("max_depth")
plt.ylabel("Accuracy")
plt.title("Accuracy by tree depth")
plt.legend()
plt.show()

**Your response:** As `max_depth` increases, what happens to training accuracy? What happens to the gap between training and test accuracy? What does the largest gap tell you about the deepest tree? (The test set is small, so test accuracy bounces around: focus on the overall pattern and on the gap.)

> Write your answer here.

## Wrap-up

In 2–3 sentences, explain when you would choose **linear regression**, **logistic regression**, or a **decision tree**. Name one strength and one weakness of a decision tree.

> Write your answer here.

### Check your code

Run the next cell to recheck all nine coding exercises.

**To submit**: Run the completed notebook from top to bottom and complete every written response. Be ready to show your mentor during class on 10/12.

In [ ]:
grader.check_all()